# Deliverable 3 — American Airlines Data Audit & EDA

**Goal:** Understand the supplied route-month passenger-demand data and airport lookup before feature engineering and forecasting.

**Confidentiality:** AA datasets are NDA-protected. Keep all raw data and executed notebook outputs local. **Clear all outputs before committing this notebook.** No predictive models are trained here; use `deliverable4/notebooks/01-model-development.ipynb` for modeling.


## 1. Load the two datasets

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root == root.parent:
        raise FileNotFoundError("Open the notebook somewhere inside the Git repository.")
    root = root.parent

demand = pd.read_csv(root / "data/raw/AA_PAX_FORECASTING.csv")
airports = pd.read_csv(root / "data/raw/AA_AIRPORT_INFO.csv")
print("Demand shape:", demand.shape)
print("Airport shape:", airports.shape)
print("Demand columns:", demand.columns.tolist())
print("Airport columns:", airports.columns.tolist())

## 2. Inspect schema and data quality

Check missing values, duplicates, and data types. `OD_PAX` is the **route-month target**; `OD_REV_USD` is same-month revenue and must not be used as an input for predicting that month.

In [ ]:
print("DEMAND TYPES")
print(demand.dtypes.to_string())
print("\nDEMAND MISSING VALUES")
print(demand.isna().sum().to_string())
print("\nDUPLICATE DEMAND ROUTE-MONTHS")
print(demand.duplicated(subset=["DEP_YEAR", "DEP_MONTH", "ORG_AIRPRT", "DES_AIRPRT"]).sum())
print("\nAIRPORT MISSING VALUES")
print(airports.isna().sum().to_string())
print("\nDUPLICATE AIRPORT CODES")
print(airports["AIRPRT_CD"].duplicated().sum())

## 3. Route coverage and time axis

In [ ]:
demand = demand.copy()
demand["ROUTE"] = demand["ORG_AIRPRT"].astype(str) + " → " + demand["DES_AIRPRT"].astype(str)
demand["DATE"] = pd.to_datetime(dict(year=demand["DEP_YEAR"], month=demand["DEP_MONTH"], day=1))

route_counts = demand.groupby("ROUTE").size()
print("Origin airports:", demand["ORG_AIRPRT"].nunique())
print("Destination airports:", demand["DES_AIRPRT"].nunique())
print("Routes:", demand["ROUTE"].nunique())
print("Date range:", demand["DATE"].min().date(), "to", demand["DATE"].max().date())
print("Distinct months in dataset:", demand["DATE"].nunique())
print("Observations per route — min/median/max:", route_counts.min(), route_counts.median(), route_counts.max())

## 4. Target distribution

A skewed distribution (mean much larger than median) suggests high-demand markets could dominate raw error metrics.

In [ ]:
print(demand["OD_PAX"].describe())
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(demand["OD_PAX"], bins=30, edgecolor="black")
ax.set(title="Distribution of Passenger Demand per Route-Month", xlabel="OD_PAX", ylabel="Route-month records")
plt.tight_layout()
plt.show()

## 5. Time trend

This is the sum over **observed rows only**. Missing routes or route-months may make the total misleading; do not interpret it as total American Airlines network demand.

In [ ]:
monthly = demand.groupby("DATE")["OD_PAX"].sum().sort_index()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.index, monthly.values, marker=".")
ax.set(title="Passenger Demand in Available Sample (Not Network-Wide)", xlabel="Month", ylabel="Sum of OD_PAX across observed rows")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Months present:", len(monthly))
print("Calendar months between endpoints:", len(pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")))

## 6. Compare recorded demand across routes

This compares totals over the available observations, **not** true route popularity across the whole network or a comparable observation period.

In [ ]:
route_totals = demand.groupby("ROUTE")["OD_PAX"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
route_totals.plot.bar(ax=ax)
ax.set(title="Total Recorded OD_PAX by Route in Provided Sample", xlabel="Route", ylabel="Sum of observed OD_PAX")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Find missing route-months

Models should use **calendar-date joins** for lags (e.g., exactly 12 months earlier) rather than `.shift(12)` unless every route has a complete monthly sequence.

In [ ]:
gap_rows = []
for route, group in demand.groupby("ROUTE"):
    observed = pd.DatetimeIndex(group["DATE"].sort_values())
    expected = pd.date_range(observed.min(), observed.max(), freq="MS")
    gap_rows.append({
        "ROUTE": route,
        "observed_months": len(observed),
        "expected_months": len(expected),
        "missing_months": len(expected.difference(observed)),
    })
gaps_df = pd.DataFrame(gap_rows)
print("Routes with missing months:", (gaps_df["missing_months"] > 0).sum())
print("Total missing route-months (within each route's start/end):", gaps_df["missing_months"].sum())
print("Min/median/max route gaps:", gaps_df["missing_months"].min(), gaps_df["missing_months"].median(), gaps_df["missing_months"].max())

## 8. Key Findings and Modeling Implications

### Data coverage
- The supplied demand file contains **1,000 monthly route records**; the airport lookup contains **1,000 records**.
- The demand sample covers **12 origin–destination routes from one origin airport**, with dates ranging from **January 2019 to July 2026**.
- All **91 calendar months** in that overall date range appear somewhere in the dataset, but **individual routes do not necessarily have complete histories**. Therefore, this sample cannot represent American Airlines' entire network.

### Data quality
- The demand table has **no null values** and **no duplicate year–month–origin–destination combinations**.
- The airport table has **no duplicate airport codes**, but some metadata is incomplete: `STATE_PROVNC_CD` has **848 missing values** and `CNTRY_CD` has **7**.
- Across the individual route histories, **5 routes have gaps**, totaling **84 missing route-month combinations** between their respective first and last observed months.

### Passenger-demand patterns
- Monthly route demand is **strongly right-skewed**: the mean (~1,085) is considerably higher than the median (~388). Large-demand observations can therefore dominate absolute error metrics.
- Recorded demand differs substantially by route. However, summing observations is not a fair measure of which route is most popular when routes have unequal coverage.
- The monthly-demand chart summarizes **only rows present in the supplied sample**. Changes in route coverage can affect the totals, so the graph must not be interpreted as company-wide demand or conclusive growth.

### Consequences for forecasting
1. **Forecast each route and month**, using `OD_PAX` as the prediction target.
2. **Split chronologically**, training on earlier months and evaluating on later unseen months.
3. **Create lag features using exact route-and-calendar-month joins**, not an unverified `shift(12)`, because several routes have missing months.
4. **Exclude same-month `OD_REV_USD` from predictors** because it would not be known when predicting that month's demand. Also exclude export row numbers and load timestamps as forecasting features.
5. **Define the forecast horizon before selecting features.** Short-horizon models and a two-year-ahead model have different information available at prediction time.
6. **Compare against a simple baseline** and evaluate both overall error (MAE/RMSE/WAPE) and route-level errors.

### Scope and limitations
These findings describe **only the supplied, geographically limited sample**. They do not establish generalizable accuracy across all American Airlines routes. The dataset is NDA-protected; review any numerical findings for permitted disclosure and **never commit raw data or executed notebook outputs**.

**Next:** Prepare reproducible, horizon-appropriate features in `02-feature-engineering.ipynb`, then train and compare models in `deliverable4/notebooks/01-model-development.ipynb`.
